## Multilayer Perceptron For Image Dataset

In [19]:
import sys
print(sys.executable)
import torch
import torch.nn as nn 

/Library/Developer/CommandLineTools/usr/bin/python3


In [20]:
# select device CPU/GPU-CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


#### Standarization/Normalization

In [21]:
# data/image/computer vision preprocessing
from torchvision import transforms # transforms provides all data/image preprocessing funtionality for Computer Vision/Matrix data/Image data
transform = transforms.Compose([
    transforms.ToTensor(), # scales pixels values from 0-255 to 0-1 [scale down to avoid computational complexity] - normalization
    transforms.Normalize(mean=(0.1307,), std=(0.3081, )) # as the dataset is benchmark, so the mean and std is given by the dataset provider for 
                                                         # pixel value 0 to 1. So, first we scale down to 0-1, the use the mean, std normalization
                                                         # Othewise, we don't need to use the normalization. 
])

#### Import and Load Dataset [Practice dataset given by torch]

In [22]:
# import torch practice datasets
from torchvision import datasets

train_dataset = datasets.MNIST(
    root='../../../../my-practice/Dataset/amnist_data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='../../../../my-practice/Dataset/amnist_data',
    train=False,
    download=True,
    transform=transform
)

In [23]:
# Load Dataset using DataLoader. DataLoader: Helps to load batch dataset.
from torch.utils.data import DataLoader
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,
    shuffle=True # When I want to distribute randomly
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=64,
    shuffle=False
)

#### Perceptron

In [24]:
class Perceptron(nn.Module):
    def __init__(self, input_size): # initialize the Perceptron using input_size and take nn.Module functionality.
        super(Perceptron, self).__init__()

        self.w = nn.Parameter(torch.randn(input_size)) # weights
        self.b = nn.Parameter(torch.randn(1)) # bias

    def forward(self, X):
        return X @ self.w + self.b # calculate prediction and return the results

In [25]:
input_size = 28 * 28 # according the dataset

#### Check sample result

In [26]:
# check sample result
perceptron = Perceptron(input_size)
sampe_input = torch.randn(input_size)
sample_output = perceptron(sampe_input)
print("Prediction = ", sample_output)

Prediction =  tensor([3.6963], grad_fn=<AddBackward0>)


#### Taking Activation Funtion [Introduced Non-linearity]

In [27]:
class RelU(nn.Module): # ReLU, result = max(0, input result of previous neuron/perceptron). Most used activation function. 
    def __init__(self):
        super(RelU, self).__init__() # initialization

    def forward(self, x):
        return torch.maximum(torch.tensor(0,0), x) # calculate result and reuturn

#### Hidden Linear Layer for multiple Perceptron

In [28]:
class Linear(nn.Module):
    def __init__(self, input_size, output_size): 
        super(Linear, self).__init__() # initialize using input and output size in this layer
        self.perceptrons = nn.ModuleList([ # define how many Perceptron will be in this layer. 
            Perceptron(input_size) for _ in range(output_size) # Input size of a singe Perceptron = total output comes from previous layer 
                                                               # total Perceptron in this layer == output_size.
        ])
    def forward(self, x): # x = input for this layer / a single output from previous layer
        outputs = [perceptron(x) for perceptron in self.perceptrons] # a single output(x) from previous layer went throuth all the perceptron in this layer. 
                                                                     # This is a input for all Perceptron in this layer
        outputs = torch.stack(outputs, dim=1) # creating matrix format from list format.  
        return outputs

#### Define Digit Classifier

In [29]:
class DigitClassifier(nn.Module):
    def __init__(self, input_size=784, output_size=10): # input size = 28*28, output = 10 digits [0 to 9]
        super(DigitClassifier, self).__init__() # Initialization
        self.fc1 = Linear(input_size, 512) # Linear(input, output) = Linear(784, 512). fc = Fully connected layer == Linear layer
        self.fc2 = Linear(512, 256) # Linear(input [previous layer output], output) = Linear (512, 256)
        self.fc3 = Linear(256, 128) # Linear(256, 128)
        self.fc4 = Linear(128, output_size) # Linear(128, 10 [final result]). Finally, we took total 4 Linear Layer to produce final prediction/output
        self.relu = nn.ReLU() # on ReLU [activation function] layer
    def forward(self, x): # x is an image for this dataset
        # print(x.shape) # B, 1, 28, 28. Where, B [Batch size], 1 [channel], 28 [H], 28 [W] - format of x [image]
        x = x.view(-1, input_size) # (B,784). Here, -1 used to conver the shape from (B, 1, 28, 28) to (B, 784)
        # How to conver: -1 does, B*1*28*28/784 = B. So, x = x.view(B, 784) and x.shape = (B, 784) - reshaped
        # print(x.shape)
        x = self.fc1(x) #  in first Linear = Linear(x) = Linear(B, 784)
        x = self.relu(x) # result went through activation function (ReLU)
        x = self.fc2(x) # result = Linear (512, 256)
        x = self.relu(x) # Relu(result)
        x = self.fc3(x) # result = (256, 128)
        x = self.relu(x) # ReLu(result)
        x = self.fc4(x) # final result/output/prediction = Linear(128, 10) - 10 digits from 0 to 9
        return x # return final result

#### Define Model

In [ ]:
""" Use of device

data is stored in (RAM/GPU) when program is running
model is also loaded (RAM/GPU)
"""
model = DigitClassifier(input_size=28 * 28, output_size=10).to(device) 
sampe_input = sampe_input.to(device) # model and sampe_input loaded in same device (GPU/RAM)

# check sample output result
output = model(sampe_input)
print(output.shape)
print(output)

torch.Size([1, 10])
tensor([[-28588.4961, -53161.4648, -71203.0469, -32321.2930, -27269.7676,
          86896.2734,   9270.3574,  20582.0605, -12890.0723,  32103.7461]],
       grad_fn=<StackBackward0>)


##### Visualized model's parameters

In [ ]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

Total trainable parameters: 567,434


In [ ]:
(784+1) * 512 + (512+1) * 256 + (256+1) * 128+ (128+1) * 10 # multiplication of all input in each layer(w) + bias(b) = Total trainable parameters

567434

In [ ]:
param_size = next(model.parameters()).element_size() # total bytes
model_size_bytes = total_params * param_size
model_size_mb = model_size_bytes / (1024 * 1024)

In [ ]:
print("\n" + "="*50)
print(f"Total trainable parameters : {total_params:,}")
print(f"Parameter size (float32)   : {param_size} bytes")
print(f"Estimated model size       : {model_size_mb:.2f} MB")
print(f"Estimated size (float16)   : {model_size_mb/2:.2f} MB")


Total trainable parameters : 567,434
Parameter size (float32)   : 4 bytes
Estimated model size       : 2.16 MB
Estimated size (float16)   : 1.08 MB


In [36]:
max_val, predicted_id = torch.max(output, 1)
print(max_val)
print("Class label:", predicted_id)

tensor([86896.2734], grad_fn=<MaxBackward0>)
Class label: tensor([5])


#### Define Optimizer and Loss/Cost Calculator

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001) # used for optimize the loss/cost
criterion = nn.CrossEntropyLoss() # Used for non-binary classification

#### Training and Evaluation

In [ ]:
num_epochs = 2

for epoch in range(num_epochs):
    model.train() # change model mode
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader): # loading a batch one by one. Here, batch_idx -> index, (data, target) -> data of the batch
        data, target = data.to(device), target.to(device) # ensures model and data are on the same device. i.e. device is of two type cpu, cuda
        outputs = model(data) # prediction
        loss = criterion(outputs, target) # calculate loss
        optimizer.zero_grad() # finding local minimum. [optimal/minimum loss points]
        loss.backward() # Compute Gradient Descent(Back Propagation)
        optimizer.step() # update weights(w) and bias(b) using optimal points
        running_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}')
    
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

Epoch [1/2], Step [1/938], Loss: 86821.7188
Epoch [1/2], Step [101/938], Loss: 6516.8730
Epoch [1/2], Step [201/938], Loss: 4018.4497
Epoch [1/2], Step [301/938], Loss: 3098.9648
Epoch [1/2], Step [401/938], Loss: 2241.2437
Epoch [1/2], Step [501/938], Loss: 1168.8379
Epoch [1/2], Step [601/938], Loss: 716.7803
Epoch [1/2], Step [701/938], Loss: 1130.4253
Epoch [1/2], Step [801/938], Loss: 1444.4124
Epoch [1/2], Step [901/938], Loss: 449.0769
Epoch 1, Loss: 3273.2934522811793
Epoch [2/2], Step [1/938], Loss: 713.2571
Epoch [2/2], Step [101/938], Loss: 546.4108
Epoch [2/2], Step [201/938], Loss: 408.4341
Epoch [2/2], Step [301/938], Loss: 185.7540
Epoch [2/2], Step [401/938], Loss: 868.0889
Epoch [2/2], Step [501/938], Loss: 467.2033
Epoch [2/2], Step [601/938], Loss: 470.9731
Epoch [2/2], Step [701/938], Loss: 975.4496
Epoch [2/2], Step [801/938], Loss: 311.8623
Epoch [2/2], Step [901/938], Loss: 262.6411
Epoch 2, Loss: 678.8313879895566


In [ ]:
model.eval() # change model mode from training [default] to evaluation
correct = 0
total = 0
with torch.no_grad(): # no need gradient calculation in evaluation part
    for data, target in test_loader: # loading test data
        data, target = data.to(device), target.to(device) # taking same device [cpu, cuda/gpu]
        outputs = model(data) # prediction
        _, predicted = torch.max(outputs.data, 1) # max predicted label
        total += target.size(0)
        correct += (predicted == target).sum().item() # see predicted label == target. Then, sum up as a correct prediction.

accuracy = 100 * correct / total # total accuracy
print(f'Accuracy on the test set: {accuracy:.2f}%')

Accuracy on the test set: 90.90%
